<a href="https://colab.research.google.com/github/nivethithanm/torchcode-solutions/blob/main/P0_08_numerical_stability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-08: Numerical Stability

**Difficulty**: 🟡 Medium

**Objective**: Learn the tricks that prevent NaN/Inf in real models. Every interview-level implementation needs these.

In [ ]:
import torch
import torch.nn.functional as F

## Task 1: Log-Sum-Exp Trick

Compute `log(sum(exp(x)))` without overflow.

In [ ]:
def log_sum_exp(x, dim=-1):
    """Numerically stable log-sum-exp.
    Trick: log(sum(exp(x))) = max(x) + log(sum(exp(x - max(x))))
    """
    # TODO: Implement
    x_max = x.max(dim=dim, keepdim=True)[0]
    log_sum_exp = x_max + torch.log(torch.sum(torch.exp(x - x_max), dim=dim, keepdim=True))
    return log_sum_exp.squeeze()

# Test with normal values
x = torch.randn(3, 5)
assert torch.allclose(log_sum_exp(x), torch.logsumexp(x, dim=-1), atol=1e-5)

# Test with extreme values (would overflow without the trick)
x_big = torch.tensor([[1000., 1001., 1002.]])
result = log_sum_exp(x_big)
assert not torch.isinf(result).any(), 'Got inf!'
assert not torch.isnan(result).any(), 'Got nan!'
print('✅ Task 1 passed!')

✅ Task 1 passed!


## Task 2: Stable Cross-Entropy

Implement cross-entropy loss using log_softmax instead of log(softmax).

In [ ]:
def stable_cross_entropy(logits, targets):
    """Numerically stable cross-entropy loss.
    Args:
        logits: (B, C) raw model outputs
        targets: (B,) class indices
    Returns: scalar loss
    """
    # TODO: Use log_softmax (not log(softmax)) then gather and negate
    log_probs = torch.log_softmax(logits, dim=-1)
    targets_2d = targets.unsqueeze(1)
    correct = log_probs.gather(dim=-1, index=targets_2d)
    loss = -correct.mean()
    return loss

logits = torch.randn(4, 10)
targets = torch.tensor([3, 7, 1, 5])
mine = stable_cross_entropy(logits, targets)
ref = F.cross_entropy(logits, targets)
assert torch.allclose(mine, ref, atol=1e-5)
print('✅ Task 2 passed!')

✅ Task 2 passed!


## Task 3: Safe Division

Normalization layers divide by std — what if std is 0?

In [ ]:
def safe_normalize(x, dim=-1, eps=1e-5):
    """Normalize x along dim, handling zero-variance case."""
    # TODO: Compute mean, var, normalize with eps in denominator
    x_std = x.std(dim=dim, keepdim=True)
    return x / (x_std + eps)

# Normal case
x = torch.randn(3, 10)
out = safe_normalize(x)
assert not torch.isnan(out).any()

# Edge case: constant input (variance = 0)
x_const = torch.ones(3, 10) * 5.0
out_const = safe_normalize(x_const)
assert not torch.isnan(out_const).any(), 'NaN with constant input!'
print('✅ Task 3 passed!')

✅ Task 3 passed!
